In [ ]:
using MPSDynamics, ChaosTools, ProgressBars
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/config/requirements.jl");

In [ ]:
const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞
ϕ_quantum=1/2

#### Caracterization of the circuit #####

ω_a = 3.8533*2*pi
ω_b_0 = 7.0515*2*pi

Lamb_shift = 0.05 #Lamb shift in GHz


φ_a = 0.11
Z_a = 2*ϕ_quantum^2*φ_a^2
C_a_inv = Z_a*ω_a
E_Ca = C_a_inv/2

φ_b = 0.204
Z_b = 2*ϕ_quantum^2*φ_b^2
C_b_inv = Z_b*ω_b_0
E_Cb = C_b_inv/2

E_J = 37.12*2*pi


##Drive


ω_d=ω_b_0#+Lamb_shift*2*pi # Drive frequency
ω_p=2ω_a-ω_d

ϵ_0 = 0.4581818181818182
g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d=10.12*g2

buffer_phase = 0.0
pump_phase = 0.0;

##Env

κ_b=1/10.4; # 1/ns from Leon's paper
κ_2 = 4*g2^2/κ_b
κ_a = 0#3e-3*κ_2 #from Leon's paper

# Check chaos for Mean-field

In [ ]:
t_max = 10/κ_b
ϕ_a_0, q_a_0=0.0, 0.0
ϕ_b_0, q_b_0=0.0, 0.0

system = SystemATS(ω_a, ω_b_0, E_J, E_Ca, E_Cb, ϕ_a_0, q_a_0, ϕ_b_0, q_b_0)
environment = MeanField(κ_a, κ_b) #OneChain(n, chain_params...)

system_prepared = prepare(system)
environment_prepared = prepare(system,environment)

dt = get_dt(system, environment)
u_dot_0, u_0 = get_init(system_prepared, environment_prepared)
u_0_new = vcat(u_0,u_dot_0);

In [ ]:
ϵ_0_list = vcat(0.5:.05:1,1:.05:0.6pi)
ϵ_d_g2_ratio_list = 2:0.3:12

In [ ]:
Lyap_coef_chaos = Array{Float64}(undef, length(ϵ_d_g2_ratio_list), length(ϵ_0_list))
drive = DriveTwoModes(0.0, ω_d, buffer_phase, 0.0, ω_p, pump_phase)
drive_prepared = prepare(drive)
Taylor_order = 5
for i in ProgressBar(1:length(ϵ_0_list))
    ϵ_0_i = ϵ_0_list[i]
    drive.ϵ_0 = ϵ_0_i
    g2_max_i = besselj(1,ϵ_0_i)*E_J*φ_a^2*φ_b # (Leon's paper)
    for j in 1:length(ϵ_d_g2_ratio_list)
        ϵ_d_j = ϵ_d_g2_ratio_list[j]*g2_max_i
        drive.ϵ_d = ϵ_d_j
        drive_prepared.drive = drive
        p = SimulationParams_Taylor(Taylor_order, system_prepared, drive_prepared, environment_prepared)

        problem = CoupledODEs(equation_1ODE_Taylor!, u_0_new, p)

        Lyap_coef_chaos[j,i] = lyapunov(problem, t_max, Δt=dt; d0 = 1e-7, d0_upper = 1e-4, Ttr = 30/κ_b)
    end
end

In [ ]:
g2_list = E_J.*besselj.(1,ϵ_0_list).*φ_a^2 .*φ_b.*1e3
g2_lis = 0:0.1:60 .*2pi
α_sq= (ϵ_d_g2_ratio_list./2)
heatmap(g2_list/2pi,α_sq,Lyap_coef_chaos)
#plot!(g2_lis/2pi, (κ*1e3./(sqrt(2)*g2_lis)).^2,label="")
plot!(g2_lis/2pi, (4*κ_b*1e3 ./g2_lis).^2,label="")
#plot!(g2_lis/2pi, (5.2*κ*1e3 ./g2_lis).^2,label="")
plot!(xlabel=L"g_2/2\pi"*" (MHz)", ylabel=L"|\alpha_{th}|^2",
    size=(size_px_car[1],size_px_car[2]),
    xlims=(minimum(g2_list/2pi),maximum(g2_list/2pi)),ylims=(minimum(α_sq),maximum(α_sq)),
    colorbar_title="\nLyapunov coef.", title = "Mean field "*L"κ_b t= "*"$(κ_b*t_max)",
    clims=(0,maximum(Lyap_coef_chaos)),
    right_margin=10mm)
#savefig("../../Figures/Two-photon/heatmap_lyapunov_coef.pdf")

In [ ]:
t_max = 10/κ_b
re_a_0, im_a_0 = 0, 0
ϕ_a_0, q_a_0=sqrt(2*Z_a)*re_a_0, sqrt(2/Z_a)*im_a_0
ϕ_b_0, q_b_0=0.0, 0.0

system = SystemATS(ω_a, ω_b_0, E_J, E_Ca, E_Cb, ϕ_a_0, q_a_0, ϕ_b_0, q_b_0)
environment = MeanField(κ_a, κ_b) #OneChain(n, chain_params...)

system_prepared = prepare(system)
environment_prepared = prepare(system,environment)

dt = get_dt(system, environment);

In [ ]:
Lyap_coef_chaos = Array{Float64}(undef, length(ϵ_d_g2_ratio_list), length(ϵ_0_list))
drive = DriveTwoModes(0.0, ω_d, buffer_phase, 0.0, ω_p, pump_phase)
drive_prepared = prepare(drive)
t_w = 30/κ_b
for i in ProgressBar(1:length(ϵ_0_list))
    ϵ_0_i = ϵ_0_list[i]
    drive.ϵ_0 = ϵ_0_i
    g2_max_i = besselj(1,ϵ_0_i)*E_J*φ_a^2*φ_b # (Leon's paper)
    for j in 1:length(ϵ_d_g2_ratio_list)
        ϵ_d_j = ϵ_d_g2_ratio_list[j]*g2_max_i
        drive.ϵ_d = ϵ_d_j
        drive_prepared.drive = drive
        re_a_0, im_a_0 = 0, sqrt(ϵ_d_g2_ratio_list[j]/2)
        ϕ_a_0, q_a_0=sqrt(2*Z_a)*re_a_0, sqrt(2/Z_a)*im_a_0
        system = SystemATS(ω_a, ω_b_0, E_J, E_Ca, E_Cb, ϕ_a_0, q_a_0, ϕ_b_0, q_b_0)
        system_prepared = prepare(system)
        u_dot_0, u_0 = get_init(system_prepared, environment_prepared)
        u_0_new = vcat(u_0,u_dot_0)
        p = SimulationParams(system_prepared, drive_prepared, environment_prepared)

        problem = CoupledODEs(equation_1ODE!, u_0_new, p)

        Lyap_coef_chaos[j,i] = lyapunov(problem, t_max, Δt=dt; d0 = 1e-7, d0_upper = 1e-4, Ttr = t_w)
    end
end

In [ ]:
Lyap_coef_chaos_sign = map(x -> x>0 ? 1 : -1, Lyap_coef_chaos);

In [ ]:
g2_list = E_J.*besselj.(1,ϵ_0_list).*φ_a^2 .*φ_b
ϵ_0_lis = 0:maximum(ϵ_0_list)/100:maximum(ϵ_0_list)
g2_lis = E_J.*besselj.(1,ϵ_0_lis[1:100]).*φ_a^2 .*φ_b
α_sq= (ϵ_d_g2_ratio_list./2)
heatmap(g2_list/2/pi*1e3,α_sq, Lyap_coef_chaos_sign, cmap=:bluesreds)
plot!(g2_lis/2/pi*1e3, (0.25./g2_lis/E_J/φ_a^2/φ_b).^2,label="")
plot!(g2_lis/2/pi*1e3, (0.25./(g2_lis.^2) .*besselj.(1,ϵ_0_lis[1:100])).^2,label="")
plot!(xlabel=L"g_2/2\pi"*" (MHz)", ylabel=L"|\alpha_{th}|^2",
    size=(size_px_car[1],size_px_car[2]),
    xlims=(minimum(g2_list/2pi*1e3),maximum(g2_list/2pi*1e3)),ylims=(minimum(α_sq),maximum(α_sq)),
    colorbar_title="\nLyapunov coef.", title = "Mean field "*L"κ_b t= "*"$(κ_b*t_max)",
    clims=(minimum(Lyap_coef_chaos_sign),maximum(Lyap_coef_chaos_sign)),
    right_margin=10mm)
#savefig("../../figures/Two-photon/heatmap_lyapunov_coef_sign.pdf")